In [1]:
import pandas as pd
import numpy as np

In [6]:
def Woe_IV_Dis(df, features, target):
    aux = features + [target] 
    
    df = df[aux].copy()
    
    # Empty dataframe
    df_woe_iv = pd.DataFrame({},index=[])
    
    for feature in features:
        df_woe_iv_aux = pd.crosstab(df[feature], df[target], normalize='columns') \
                        .assign(RR=lambda dfx: dfx[1] / dfx[0]) \
                        .assign(WoE=lambda dfx: np.log(dfx[1] / dfx[0])) \
                        .assign(IV=lambda dfx: (dfx['WoE']*(dfx[1]-dfx[0]))) \
                        .assign(IV_total=lambda dfx: np.sum(dfx['IV']))

        df_woe_iv = pd.concat([df_woe_iv, df_woe_iv_aux])
    
    return df_woe_iv  

In [2]:
df = pd.read_csv('df_comp2.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4501860 entries, 0 to 4501859
Data columns (total 36 columns):
 #   Column                                     Dtype  
---  ------                                     -----  
 0   Unnamed: 0                                 int64  
 1   n_coord                                    int64  
 2   ocorrencia                                 float64
 3   vintequatrohoras                           float64
 4   Altitude_numerica                          float64
 5   Declividade_numerica                       float64
 6   graurisc                                   float64
 7   lon_ocr                                    float64
 8   lat_ocr                                    float64
 9   Orientacao_numerica                        float64
 10  Curv_Vertical_numerica                     float64
 11  Curv_Horizontal_numerica                   float64
 12  Relevo_sombreado_numerico                  float64
 13  Vegetacao_Natural_Dominante               

In [13]:
discretas = ['flg_exploracao_mineral'                     
,'flg_rocha'                                  
,'flg_cobertura_vegetal'                      
,'flg_afloramento_rochoso'                    
,'flg_ocupacao_desordenada'  ]

continuas = ['vintequatrohoras'         
,'Altitude_numerica'        
,'Declividade_numerica'     
,'graurisc'                 
,'lon_ocr'                  
,'lat_ocr'                  
,'Orientacao_numerica'      
,'Curv_Vertical_numerica'   
,'Curv_Horizontal_numerica' 
,'Relevo_sombreado_numerico']

In [14]:
Woe_IV_Dis(df, discretas, 'ocorrencia')

/home/pdpa-chuvas/miniconda3/envs/tf/lib/python3.8/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


ocorrencia,0.0,1.0,RR,WoE,IV,IV_total
0,0.998405,0.998856,1.000452,0.000451,2.035166e-07,0.000150
1,0.001595,0.001144,0.717352,-0.332189,1.497570e-04,0.000150
0,1.000000,1.000000,1.000000,0.000000,0.000000e+00,0.000000
0,0.508371,0.927918,1.825275,0.601730,2.524537e-01,1.057946
1,0.491629,0.072082,0.146620,-1.919914,8.054924e-01,1.057946
0,0.979581,1.000000,1.020844,0.020630,4.212361e-04,inf
1,0.020419,0.000000,0.000000,-inf,inf,inf
0,0.977037,0.956522,0.979002,-0.021221,4.353746e-04,0.013533
1,0.022963,0.043478,1.893444,0.638398,1.309720e-02,0.013533


In [15]:
def Woe_IV_cont(df, features, target):
    
    aux = features + [target] 
    
    df = df[aux].copy()
    
    # Empty dataframe
    df_woe_iv = pd.DataFrame({},index=[])
    
    # Number of rows with target = 1
    _t1 = sum(df[target])
    # Number of rows with target = 0
    _t0 =  len(df[target]) - _t1
    
    # Percentile of continuous variables
    _quantile = df.iloc[:, :-1].quantile([.1, .2, .3, .4, .5, .6, .7, .8, .9], axis = 0)
    
    
    for _column in _quantile.columns:
   
        # Non-duplicated quantile limit values
        list_aux = _quantile[[_column]].drop_duplicates().to_numpy()
    
        _tiv = 0
        
        for q in range(len(list_aux)):
            
            
            if q>0:
                location = df[(df.loc[:,_column] > float(list_aux[q-1])) & (df.loc[:,_column] <= float(list_aux[q]))].index
                limit = str(list_aux[q-1]) + ' a ' + str(list_aux[q])
            else:
                location = df[(df.loc[:,_column] <= float(list_aux[q]))].index
                limit = '<=' + str(list_aux[q])
                
            _many = len(location)  
            
            # Target = 1
            _1 = sum(df.loc[location,target])
            _p1 = _1/_t1
            
            # Target = 0
            _0 = _many - _1
            _p0 = _0/_t0
            
            # Relative risk
            if _p1 == 0 or _p0 == 0:
                _RR = 1
            else:
                _RR = _p1/_p0
            
            # Weight of evidence
            _woe = np.log(_RR)
            
            # Information value
            _iv = round(_woe*(_p1-_p0),2)
            
            # Information value - total
            _tiv = _tiv+_iv
                    
            
            dframe = pd.DataFrame({'variable': _column , 'limit':limit , '0': _p0 , '1': _p1, 'RR':_RR, 'WoE': _woe , 'IV':  _iv}
                                  , index = [ _column])  
            
            df_woe_iv = pd.concat([df_woe_iv, dframe], ignore_index=True)
            
        dframe = pd.DataFrame({'variable': _column ,'limit': ' ' , '0': 1 , '1': 1, 'RR': 1 , 'WoE': 0 , 'IV':  _tiv}
                                  , index =[ _column])
         
        df_woe_iv = pd.concat([df_woe_iv, dframe], ignore_index=True)
            
    return df_woe_iv

In [18]:
pd.set_option('display.max_rows', None)

In [19]:
Woe_IV_cont(df, continuas, 'ocorrencia')

,variable,limit,0,1,RR,WoE,IV
0,vintequatrohoras,<=[-11.51292546],0.655189,0.382151,0.583269,-0.539107,0.15
1,vintequatrohoras,[-11.51292546] a [-0.91629073],0.057257,0.033181,0.579506,-0.545579,0.01
2,vintequatrohoras,[-0.91629073] a [1.02961942],0.089525,0.091533,1.022428,0.022181,0.00
3,vintequatrohoras,[1.02961942] a [2.4510051],0.098115,0.123570,1.259445,0.230671,0.01
4,vintequatrohoras,,1.000000,1.000000,1.000000,0.000000,0.17
5,Altitude_numerica,<=[3.59105736],0.100158,0.108696,1.085244,0.081805,0.00
6,Altitude_numerica,[3.59105736] a [3.93789283],0.099830,0.154462,1.547254,0.436482,0.02
7,Altitude_numerica,[3.93789283] a [4.24718513],0.100149,0.155606,1.553753,0.440673,0.02
8,Altitude_numerica,[4.24718513] a [4.47266383],0.099833,0.138444,1.386755,0.326967,0.01
9,Altitude_numerica,[4.47266383] a [4.61568794],0.100160,0.098398,0.982411,-0.017745,0.00
